In [ ]:
import sys
import traceback
import logging
import portalocker
import os

# 경로 설정 - 플랫폼에 따라 다르게 처리
if os.name == 'nt':  # Windows
    sys.path.insert(0, r"Y:/git/pyaedt_library/src/")
else:  # Linux/Unix
    # Linux 서버 경로들 시도
    possible_paths = [
        # r"/gpfs/home1/r1jae262/jupyter/git/pyaedt_library/src/",
        r"../pyaedt_library/src/",
        os.path.join(os.path.dirname(__file__), "../git/pyaedt_library/src/"),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            sys.path.insert(0, path)
            break

import pyaedt_module
from pyaedt_module.core import pyDesktop
import os
import time
from datetime import datetime

import math
import copy

import pandas as pd

import platform
import csv

from scipy.signal import find_peaks

from module.variable import set_variable




class Simulation() :

    def __init__(self, desktop=None) :

        self.NUM_CORE = 4
        self.NUM_TASK = 1

        # Desktop은 바깥에서 생성(with 포함)해서 주입하는 것을 권장
        # (루프 중 프로젝트 close/delete가 Desktop 핸들을 무효화시키는 문제를 줄이기 위함)
        self.desktop = desktop

        

    def create_simulation_name(self):

        file_path = "./simulation_num.txt"

        # 파일이 존재하지 않으면 생성
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as file:
                file.write("1")

        # 읽기/쓰기 모드로 파일 열기
        with open(file_path, "r+", encoding="utf-8") as file:
            # 파일 잠금: LOCK_EX는 배타적 잠금,  블로킹 모드로 실행 (Windows/Linux 호환)
            portalocker.lock(file, portalocker.LOCK_EX)

            # 파일에서 값 읽기
            content = int(file.read().strip())
            self.num = content
            self.PROJECT_NAME = f"simulation{content}"
            content += 1

            # 파일 포인터를 처음으로 되돌리고, 파일 내용 초기화 후 새 값 쓰기
            file.seek(0)
            file.truncate()
            file.write(str(content))

    def create_simulation_name(self):

        file_path = "./simulation_num.txt"

        # 파일이 존재하지 않으면 생성
        if not os.path.exists(file_path):
            with open(file_path, "w", encoding="utf-8") as file:
                file.write("1")

        # 읽기/쓰기 모드로 파일 열기
        with open(file_path, "r+", encoding="utf-8") as file:
            # 파일 잠금: LOCK_EX는 배타적 잠금,  블로킹 모드로 실행 (Windows/Linux 호환)
            portalocker.lock(file, portalocker.LOCK_EX)

            # 파일에서 값 읽기
            content = int(file.read().strip())
            self.num = content
            self.PROJECT_NAME = f"simulation{content}"
            content += 1

            # 파일 포인터를 처음으로 되돌리고, 파일 내용 초기화 후 새 값 쓰기
            file.seek(0)
            file.truncate()
            file.write(str(content))

    def set_variable(self, design):
        return set_variable(self, design)



In [ ]:
GUI = False
desktop = None

desktop = pyDesktop(version=None, non_graphical=GUI, close_on_exit=False, new_desktop=True)

sim1 = Simulation(desktop=desktop)

sim1.create_simulation_name()

# simulation 디렉토리 생성 (존재하지 않으면)
simulation_dir = "./simulation"
if not os.path.exists(simulation_dir):
    os.makedirs(simulation_dir, exist_ok=True)

# 절대 경로로 변환
project_path = os.path.abspath(os.path.join(simulation_dir, sim1.PROJECT_NAME))

# desktop이 None이거나 유효하지 않은지 확인
if sim1.desktop is None:
    raise RuntimeError("Desktop instance is None. Cannot create project.")

try:
    project1 = sim1.desktop.create_project(path=project_path, name=sim1.PROJECT_NAME)
except Exception as e:
    error_msg = f"Failed to create project '{sim1.PROJECT_NAME}' at path '{project_path}': {e}\n"
    print(error_msg, file=sys.stderr)
    sys.stderr.flush()
    raise

design1 = project1.create_design(name="IPMSM", solver="Maxwell 2D", solution="Transient")

sim1.project = project1

input_data = sim1.set_variable(design1)


stator = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_outer_radius", num_sides=72, name="stator", material="copper")
stator_sub = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_outer_radius - stator_back_yoke_thick", num_sides=72, name="stator_sub", material="copper")
design1.modeler.subtract(blank_list=stator, tool_list=[stator_sub], keep_originals=False)

teeth = design1.modeler.create_rectangle(origin=["stator_outer_radius - stator_back_yoke_thick", "-stator_teeth_width/2", 0], sizes=["-stator_teeth_length", "stator_teeth_width"], name="stator_teeth", material="copper")
design1.modeler.duplicate_around_axis(assignment=[teeth], axis="Z", angle="(360/slot_num)deg", clones="slot_num", create_new_objects=False)

stator_gap = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_inner_radius + stator_gap", num_sides=72, name="stator_gap", material="copper")
stator_gap_sub = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_inner_radius", num_sides=72, name="stator_gap_sub", material="copper")
design1.modeler.subtract(blank_list=stator_gap, tool_list=[stator_gap_sub], keep_originals=False)

slot_opening = design1.modeler.create_rectangle(origin=[0, "-slot_opening/2", 0], sizes=["stator_outer_radius", "slot_opening"], name="slot_opening", material="copper")
design1.modeler.rotate(assignment=[slot_opening], axis="Z", angle="(360/slot_num/2)deg")
design1.modeler.duplicate_around_axis(assignment=[slot_opening], axis="Z", angle="(360/slot_num)deg", clones="slot_num", create_new_objects=False)
design1.modeler.subtract(blank_list=stator_gap, tool_list=[slot_opening], keep_originals=False)

In [ ]:
stator = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_outer_radius", num_sides=72, name="stator", material="copper")
stator_sub = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_outer_radius - stator_back_yoke_thick", num_sides=72, name="stator_sub", material="copper")
design1.modeler.subtract(blank_list=stator, tool_list=[stator_sub], keep_originals=False)

teeth = design1.modeler.create_rectangle(origin=["stator_outer_radius - stator_back_yoke_thick", "-stator_teeth_width/2", 0], sizes=["-stator_teeth_length", "stator_teeth_width"], name="stator_teeth", material="copper")
design1.modeler.duplicate_around_axis(assignment=[teeth], axis="Z", angle="(360/slot_num)deg", clones="slot_num", create_new_objects=False)

stator_gap = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_inner_radius + stator_gap", num_sides=72, name="stator_gap", material="copper")
stator_gap_sub = design1.modeler.create_circle(origin=[0, 0, 0], radius="stator_inner_radius", num_sides=72, name="stator_gap_sub", material="copper")
design1.modeler.subtract(blank_list=stator_gap, tool_list=[stator_gap_sub], keep_originals=False)

slot_opening = design1.modeler.create_rectangle(origin=[0, "-slot_opening/2", 0], sizes=["stator_outer_radius", "slot_opening"], name="slot_opening", material="copper")
design1.modeler.rotate(assignment=[slot_opening], axis="Z", angle="(360/slot_num/2)deg")
design1.modeler.duplicate_around_axis(assignment=[slot_opening], axis="Z", angle="(360/slot_num)deg", clones="slot_num", create_new_objects=False)
design1.modeler.subtract(blank_list=stator_gap, tool_list=[slot_opening], keep_originals=False)

In [ ]:
stator.id